# task 4 -> safety calibration of the aligned policies

## imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.data import load_yaml, read_jsonl, repo_path

## global env variables ->

In [ ]:
#papermill overrides CONFIG for the smoke test, the defaults are the real run
CONFIG = 'configs/feedback.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir']) / 'task4_safety'
FIGURES = RESULTS / 'figures'

POLICIES = ['sft', 'dpo', 'ppo', 'grpo'] #the four fixed policies, sft = the untouched base model
LABELS = ['SAFE_ANSWER', 'JUSTIFIED_REFUSAL', 'OVER_REFUSAL', 'UNSAFE_COMPLIANCE', 'AMBIGUOUS']
#one fixed colour per behaviour label -> the two wanted behaviours in blue/green, the two failures in amber/red, ambiguous grey
LABEL_COLORS = {'SAFE_ANSWER': '#2a78d6', 'JUSTIFIED_REFUSAL': '#1baf7a', 'OVER_REFUSAL': '#eda100',
                'UNSAFE_COMPLIANCE': '#d03b3b', 'AMBIGUOUS': '#c3c2b7'}

PYTHON = sys.executable #the interpreter running this notebook, so the scripts use the same environment

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True})


def run(command):
    #runs one of the release's entry points as its own process, exactly as it would be typed in a terminal.
    #each script loads its own models and writes its results to disk, nothing is passed through notebook state
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'

## common safety evaluation ->

one greedy response per policy for every xstest prompt, same decoding and the same 256 token cap

In [ ]:
run(f'{PYTHON} -m task4_safety.generate_responses --config {CONFIG}')

## categorical ai scoring ->

the fixed judge gives every (prompt, response) one of the five behaviour labels

In [ ]:
run(f'{PYTHON} -m task4_safety.judge_responses --config {CONFIG}')

## manual audit sheet ->

the blind sheet for the 60 fixed prompts, written once. it is labelled by hand, the scripts never fill it

In [ ]:
run(f'{PYTHON} -m task4_safety.make_audit_sheet --config {CONFIG}')

## aggregation

In [ ]:
#pure table work, no model. safe to run again after the audit sheet has been labelled, it then adds the agreement tables
if RUN_SCRIPTS or (RESULTS / 'judged_sft.jsonl').exists():
    get_ipython().system(f'{PYTHON} -m task4_safety.evaluate_safety --config {CONFIG}')
    assert get_ipython().user_ns['_exit_code'] == 0

## safety-calibration comparison

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
judged = pd.concat([pd.DataFrame(read_jsonl(RESULTS / f'judged_{policy}.jsonl')) for policy in POLICIES], ignore_index=True)
calibration = pd.read_csv(RESULTS / 'safety_calibration.csv').set_index('policy')
display(calibration[['safe_answer_rate', 'safe_over_refusal_rate', 'unsafe_compliance_rate', 'justified_refusal_rate',
                     'ambiguous_rate', 'mean_response_tokens']].round(4))
display(calibration.drop(columns=['safe_answer_rate', 'safe_over_refusal_rate', 'unsafe_compliance_rate',
                                  'justified_refusal_rate', 'ambiguous_rate', 'mean_response_tokens']).round(4))

In [ ]:
#judge-label make-up of each policy's responses, safe prompts and unsafe prompts side by side. one bar per policy,
#the segments are shares of that prompt class and add up to 1
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), sharey=True)
for axis, benchmark_class in zip(axes, ['SAFE', 'UNSAFE']):
    frame = judged[judged['benchmark_class'] == benchmark_class]
    shares = pd.crosstab(frame['policy'], frame['judge_label'], normalize='index').reindex(index=POLICIES, columns=LABELS, fill_value=0)
    left = np.zeros(len(POLICIES))
    for label in LABELS:
        bars = axis.barh(POLICIES, shares[label], left=left, color=LABEL_COLORS[label], edgecolor='#fcfcfb', linewidth=2, label=label)
        for bar, share in zip(bars, shares[label]):
            if share >= 0.04: #only segments wide enough to hold the number are labelled
                axis.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2, f'{share:.2f}',
                          ha='center', va='center', color='#0b0b0b', fontsize=9)
        left += shares[label].to_numpy()
    axis.set_title(f'{benchmark_class.lower()} prompts (n = {frame["xstest_id"].nunique()})')
    axis.set_xlabel('share of prompts')
    axis.set_xlim(0, 1)
    axis.grid(axis='y', visible=False)
axes[0].invert_yaxis()
axes[1].legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
fig.savefig(FIGURES / 'label_distribution.png', dpi=200)
plt.show()

## category-level behaviour

In [ ]:
#the full judge-label distribution per xstest category and policy
category = pd.read_csv(RESULTS / 'category_labels.csv')
display(category[['benchmark_class', 'type', 'policy', *LABELS, 'n']])

#the failure rate that matters for each side, per category -> over-refusal on the safe categories, unsafe compliance
#on the unsafe (contrast) categories. darker = more failures
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for axis, (benchmark_class, column, title) in zip(axes, [('SAFE', 'OVER_REFUSAL_share', 'over-refusal rate, safe categories'),
                                                          ('UNSAFE', 'UNSAFE_COMPLIANCE_share', 'unsafe-compliance rate, unsafe categories')]):
    grid = (category[category['benchmark_class'] == benchmark_class].pivot(index='type', columns='policy', values=column)[POLICIES])
    image = axis.imshow(grid.to_numpy(), cmap='Blues', vmin=0, vmax=max(0.2, grid.to_numpy().max()), aspect='auto')
    for row in range(grid.shape[0]):
        for column_index in range(grid.shape[1]):
            value = grid.iat[row, column_index]
            axis.text(column_index, row, f'{value:.2f}', ha='center', va='center',
                      color='#fcfcfb' if value > 0.6 * image.get_clim()[1] else '#0b0b0b', fontsize=9)
    axis.set_xticks(range(len(POLICIES)), POLICIES)
    axis.set_yticks(range(len(grid)), grid.index)
    axis.set_title(title)
    axis.grid(False)
fig.tight_layout()
fig.savefig(FIGURES / 'category_failure_rates.png', dpi=200)
plt.show()

## preference / reward scores against safety calibration

In [ ]:
#the held-out learned-reward score each policy reached in its own task, next to its safety rates. the reward numbers
#come from each task's own evaluation set and decoding, so they are compared across rows only as before/after pairs
#(each task's reference row against its trained row), not as one scale
sources = {'dpo': ('task1_dpo', 'eval_base.json', 'eval_standard.json'),
           'ppo': ('task2_ppo', 'eval_reference.json', 'eval_standard.json'),
           'grpo': ('task3_grpo', 'eval_reference.json', 'eval_standard.json')}
rows = []
for policy, (folder, reference_file, trained_file) in sources.items():
    base = repo_path(cfg['results_dir']) / folder
    if (base / trained_file).exists() and (base / reference_file).exists():
        reference, trained = json.loads((base / reference_file).read_text()), json.loads((base / trained_file).read_text())
        rows.append({'policy': policy, 'reward_reference': reference['reward_mean'], 'reward_trained': trained['reward_mean'],
                     'reward_change': trained['reward_mean'] - reference['reward_mean'],
                     'preference_accuracy': trained.get('heldout_preference_accuracy')})
if rows:
    reward = pd.DataFrame(rows).set_index('policy')
    change = calibration[['safe_answer_rate', 'safe_over_refusal_rate', 'unsafe_compliance_rate', 'justified_refusal_rate']]
    change = (change.loc[reward.index] - change.loc['sft']).add_suffix('_vs_sft') #each policy's safety rates relative to the sft baseline
    display(reward.join(change).round(4))
else:
    print('task 1-3 evaluation files not found next to these results, skipping')

## manual audit agreement

In [ ]:
#filled only after results/task4_safety/manual_audit_sheet.csv has been labelled by hand and evaluate_safety re-run
if (RESULTS / 'audit_agreement.csv').exists():
    display(pd.read_csv(RESULTS / 'audit_agreement.csv').round(4))
    print('rows = manual label, columns = judge label')
    display(pd.read_csv(RESULTS / 'audit_confusion.csv', index_col=0))
else:
    sheet = pd.read_csv(RESULTS / 'manual_audit_sheet.csv')
    print(f'manual audit sheet not labelled yet -> {len(sheet)} distinct responses waiting in {RESULTS / "manual_audit_sheet.csv"}')

## qualitative candidates ->

shortlists to read through and pick the report examples from, nothing here is selected automatically

In [ ]:
pd.set_option('display.max_colwidth', 400)

#prompts on which the four policies were given different judge labels, one block per behaviour the manual asks for.
#each row shows all four labels, followed by the responses of the policies that got the label in question
disagreements = pd.read_csv(RESULTS / 'policy_disagreements.csv')
print(f'{len(disagreements)} prompts with differing judge labels across the policies')
responses = judged.pivot(index='xstest_id', columns='policy', values='response')

for label in ['UNSAFE_COMPLIANCE', 'JUSTIFIED_REFUSAL', 'OVER_REFUSAL']:
    hits = disagreements[(disagreements[POLICIES] == label).any(axis=1)].head(4)
    print(f'\n{label}: {len(disagreements[(disagreements[POLICIES] == label).any(axis=1)])} prompts where at least one but not all policies got this label')
    display(hits[['xstest_id', 'benchmark_class', 'type', 'prompt', *POLICIES]])
    for _, row in hits.head(2).iterrows():
        display(pd.DataFrame({'judge_label': row[POLICIES], 'response': responses.loc[row['xstest_id'], POLICIES]}))

In [ ]:
#audited responses where the hand label and the judge disagree -> the judge-error candidates
if (RESULTS / 'audit_disagreements.csv').exists():
    audit_disagreements = pd.read_csv(RESULTS / 'audit_disagreements.csv')
    print(f'{len(audit_disagreements)} audited policy responses where manual and judge labels differ')
    display(audit_disagreements[['xstest_id', 'policy', 'benchmark_class', 'type', 'manual_label', 'judge_label',
                                 'judge_confidence', 'judge_rationale', 'prompt', 'response']].head(12))